In [0]:
SILVER_PATH = "/Volumes/workspace/legal_data/silver/legal_sections/"

In [0]:
import re
import uuid
from pyspark.sql.functions import col
from pyspark.sql import Row

In [0]:
BRONZE_PATH = "/Volumes/workspace/legal_data/bronze/legal_documents/"

bronze_df = spark.read.format("delta").load(BRONZE_PATH)

bronze_df = bronze_df.filter(col("status") == "success")
bronze_df.count()

61

In [0]:
def clean_legal_text(text):
    if not text:
        return ""

    # remove page numbers
    text = re.sub(r'Page\s*\d+', ' ', text, flags=re.IGNORECASE)

    # remove headers/footers repeated in uppercase
    text = re.sub(r'\b[A-Z ]{6,}\b', ' ', text)

    # remove bullet symbols
    text = re.sub(r'[•●■▪]', ' ', text)

    # normalize whitespace
    text = re.sub(r'\n\s*\n', '\n', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [0]:
def extract_act_name(file_name, category):
    name = file_name.replace(".pdf", "")
    name = name.replace("_", " ").replace("-", " ")

    if "constitution" in name.lower():
        return "Constitution of India"

    return name.title()

In [0]:
LEGAL_PATTERNS = [
    r'(Article\s+\d+[A-Za-z\-]*)',
    r'(Section\s+\d+[A-Za-z\-]*)',
    r'(Sec\.?\s*\d+[A-Za-z\-]*)',
    r'(Rule\s+\d+[A-Za-z\-]*)',
    r'(Chapter\s+[IVXLC]+)',
]

In [0]:
def split_legal_sections(text):
    if not text:
        return []

    pattern = "|".join(LEGAL_PATTERNS)
    parts = re.split(pattern, text)

    sections = []

    for i in range(1, len(parts), 2):
        header = parts[i]
        content = parts[i+1] if i+1 < len(parts) else ""

        # ✅ SAFETY CHECKS
        if header is None:
            continue

        header = str(header).strip()
        content = str(content).strip() if content else ""

        if header:  # ensure header not empty
            sections.append((header, content))

    return sections

In [0]:
def split_paragraphs(text, max_chars=1200):
    paragraphs = re.split(r'\n+', text)

    chunks = []
    current = ""

    for para in paragraphs:
        if len(current) + len(para) < max_chars:
            current += " " + para
        else:
            chunks.append(current.strip())
            current = para

    if current:
        chunks.append(current.strip())

    return chunks

In [0]:
records = []

for row in bronze_df.collect():

    cleaned_text = clean_legal_text(row.raw_text)
    act_name = extract_act_name(row.file_name, row.category)

    sections = split_legal_sections(cleaned_text)

    # CASE 1: Sections found
    if sections:
        for header, body in sections:
            records.append(Row(
                section_id=str(uuid.uuid4()),
                doc_id=row.doc_id,
                file_name=row.file_name,
                category=row.category,
                act_name=act_name,
                section_number=header,
                section_title=header,
                section_text=body,
                source_path=row.file_path,
                created_at=row.ingestion_time
            ))

    # CASE 2: No sections → fallback paragraph chunks
    else:
        paragraphs = split_paragraphs(cleaned_text)

        for para in paragraphs:
            records.append(Row(
                section_id=str(uuid.uuid4()),
                doc_id=row.doc_id,
                file_name=row.file_name,
                category=row.category,
                act_name=act_name,
                section_number=None,
                section_title=None,
                section_text=para,
                source_path=row.file_path,
                created_at=row.ingestion_time
            ))

In [0]:
silver_df = spark.createDataFrame(records)

silver_df.display()

section_id doc_id file_name category act_name section_number section_title section_text source_path created_at 4e00c542-5550-429f-b4d4-3619f6ab6ff8 364cfa16-7717-4ebb-b565-3c4cfd49cd16 Aadhaar_Act_2016.pdf acts Aadhaar Act 2016 null null /Volumes/workspace/legal_data/raw_documents/legal_datasets/acts/Aadhaar_Act_2016.pdf 2026-02-26T12:55:39.214Z e092722b-7428-43ae-8268-1986c19c02b2 364cfa16-7717-4ebb-b565-3c4cfd49cd16 Aadhaar_Act_2016.pdf acts Aadhaar Act 2016 null null ( , ) ACT, 2016 ______ ______ 1. Short title, extent and commencement. 2. Definitions. 3. Aadhaar number. 3A. Aadhaar number of children. 4. Properties of Aadhaar number. 5. Special measures for issuance of Aadhaar number to certain category of persons. 6. Update of certain information. 7. Proof of Aadhaar number necessary for receipt of certain subsidies, benefits and services, etc. 8. Authentication of Aadhaar number. 8A. Offline verification of Aadhaar number. 9. Aadhaar number not evidence of citizenship or domicile, etc. 10. Central Identities Data Repository. 11. Establishment of Authority. 12. Composition of Authority. 13. Qualifications for appointment of Chairperson and Members of Authority. 14. Term of office and other conditions of service of Chairperson and Members. 15. Removal of Chairperson and Members. 16. Restrictions on Chairperson or Members on employment after cessation of office. 17. Functions of Chairperson. 1 18. Chief executive officer. 19. Meetings of Authority. 20. Vacancies, etc., not to invalidate proceedings of Authority. 21. Officers and other employees of Authority. 22. Transfer of assets, liabilities of Authority. 23. Powers and functions of Authority. 23A. Power of Authority to issue directions. , 24. Grants by Central Government. 25. Fund. 26. Accounts and audit. 27. Returns and annual report, etc. 28. Security and confidentiality of information. 29. Restriction on sharing information. 30. Biometric information deemed to be sensitive personal information. 31. Alteration of demographic information or biometric information. 32. Access to own information and records of requests for authentication. 33. Disclosure of information in certain cases. 33A. Penalty for failure to comply with provisions of this Act, rules, regulations and directions. 33B. Power to adjudicate. 33C. Appeals to Appellate Tribunal. 33D. Procedure and powers of the Appellate Tribunal 33E. Appeal to Supreme Court of India. 33F. Civil court not to have jurisdiction. 2 34. Penalty for impersonation at time of enrolment. 35. Penalty for impersonation of Aadhaar number holder by changing demographic information or biometric information. 36. Penalty for impersonation. 37. Penalty for disclosing identity information. 38. Penalty for unauthorised access to the Central Identities Data Repository. 39. Penalty for tampering with data in Central Identities Data Repository. 40. Penalty for unauthorised use by requesting entity or offline verification-seeking entity. 41. Penalty for non-compliance with intimation requirements. 42. General penalty. 43. Offences by companies. 44. Act to apply for offence or contravention committed outside India. 45. Power to investigate offences. 46. Penalties not to interfere with other punishments. 47. Cognizance of offences. 48. Power of Central Government to supersede Authority. 49. Members, officers, etc., to be public servants. 50. Power of Central Government to issue directions. 50A. Exemption from tax on income. 51. Delegation. 52. Protection of action taken in good faith. 53. Power of Central Government to make rules. 54. Power of Authority to make regulations. 55. Laying of rules and regulations before Parliament. 56. Application of other laws not barred. 57. [Omitted.]. 58. Power to remove difficulties. 59. Savings. 3 ( , ) ACT, 2016 . 18 OF 2016 [25th March, 2016.] An Act to provide for, as a good governance, efficient, transparent, and targeted delivery of subsidies, benefits and services, the expenditure for which is incurred from

In [0]:
silver_df.write.format("delta") \
    .mode("overwrite") \
    .save(SILVER_PATH)

In [0]:
%sql
CREATE TABLE workspace.default.silver_legal_sections (
  section_id STRING,
  doc_id STRING,
  file_name STRING,
  category STRING,
  act_name STRING,
  section_number STRING,
  section_title STRING,
  section_text STRING,
  source_path STRING,
  created_at TIMESTAMP
)
USING DELTA;

In [0]:
silver_df.write \
    .mode("append") \
    .saveAsTable("workspace.default.silver_legal_sections")

In [0]:
silver_df.count()

1281

In [0]:
silver_df.select('file_name').distinct().show()


+--------------------+
|           file_name|
+--------------------+
|Companies Act_201...|
|the-competition-a...|
|Aadhaar_Act_2016.pdf|
|Consumer Protecti...|
|Digital Personal ...|
|Prevention of Cor...|
|the_code_of_crimi...|
|the_code_of_civil...|
|Hindu Marriage Ac...|
|special_marriage_...|
|Negotiable Instru...|
|Environment Prote...|
|Information Techn...|
|Transfer of Prope...|
|Essential Commodi...|
|Limitation Act_19...|
|Right to Informat...|
|the_arbitration_a...|
|Indian Contract A...|
|The Constitution ...|
+--------------------+
only showing top 20 rows


In [0]:
silver_df.groupBy("file_name").count().display()

file_name,count
Companies Act_2013.pdf,22
the-competition-act_2003.pdf,5
Aadhaar_Act_2016.pdf,2
Consumer Protection Act_2019.pdf,3
Digital Personal Data Protection Act_2023.pdf,12
Prevention of Corruption Act_1988.pdf,3
the_code_of_criminal_procedure_1973.pdf,56
the_code_of_civil_procedure_1908.pdf,10
Hindu Marriage Act_1955.pdf,4
special_marriage_act.pdf,25


In [0]:
silver_df.select("section_number","section_title").limit(10).display()

section_number,section_title
null,null
null,null
Chapter XX,Chapter XX
Sec. 22,Sec. 22
Chapter III,Chapter III
Chapter IV,Chapter IV
Chapter IV,Chapter IV
Chapter III,Chapter III
Chapter V,Chapter V
Chapter XIV,Chapter XIV


In [0]:
silver_df.select("section_text").limit(5).display()

section_text
""
""


In [0]:
%sql
select count(*) from workspace.default.silver_legal_sections

count(*)
1281


In [0]:
%sql
select file_name, category, act_name from workspace.default.silver_legal_sections limit 25

file_name,category,act_name
supreme court landmark judgments_02.pdf,judgments,Supreme Court Landmark Judgments 02
supreme court landmark judgments_02.pdf,judgments,Supreme Court Landmark Judgments 02
supreme court landmark judgments_02.pdf,judgments,Supreme Court Landmark Judgments 02
supreme court landmark judgments_02.pdf,judgments,Supreme Court Landmark Judgments 02
supreme court landmark judgments_02.pdf,judgments,Supreme Court Landmark Judgments 02
supreme court landmark judgments_02.pdf,judgments,Supreme Court Landmark Judgments 02
supreme court landmark judgments_02.pdf,judgments,Supreme Court Landmark Judgments 02
supreme court landmark judgments_02.pdf,judgments,Supreme Court Landmark Judgments 02
supreme court landmark judgments_02.pdf,judgments,Supreme Court Landmark Judgments 02
supreme court landmark judgments_02.pdf,judgments,Supreme Court Landmark Judgments 02
